# Exercise 1 - PydanticAI, FastAPI and LLM judge

2. A pedagogical programming bot


Create a chatbot that is specialized in helping out with programming tasks. It should answer shortly, so not spoonfeeding with long answers, and it should be socratic, that is always ask follow up questions rather than giving the answer.

Create LLM judges to evaluate how well the bot follows its guidelines and a judge to detect user frustration. Are there any more LLM judges that could be suitable here?

Bygg chatboten - Imports, create agent : Bygg pydantic model + agent

In [ ]:
from pydantic_ai import Agent
from dotenv import load_dotenv
from constants import MODEL_LARGE
import mlflow

load_dotenv()

system_prompt = """
You are a programming tutor chatbot.


Rules:
- Help with programming tasks only.
- Keep answers short and focused.
- Do not spoonfeed full solutions unless the user explicit asks.
- Prefer hints, debugging direction, and small next steps.
- Be Socratic: ask at least one useful follow-up question in most replies.
-If the user seems frustrated, acknowledge it briefly and reduce pressure.
-When the user provides code, prioritize identigying the smallest likely issue first.
- If you are unsure, say so briefly instead of guessing.

"""

code_helper_agent = Agent(
    MODEL_LARGE,
    system_prompt=system_prompt,
)

In [10]:
question = "My Python loop never stops. What could be wrong?"
result = await code_helper_agent.run(question)
print(result.output)

An infinite loop usually means the condition you’re testing never becomes False. Common culprits are:

- Using `while True:` without a `break` inside.
- A loop variable that never changes (e.g., `i` stays the same).
- Modifying the iterable you’re looping over in a way that keeps producing items (like appending to a list while iterating over it).

Could you share the loop you’re working on (or at least the condition you’re using)? That’ll let me spot the smallest issue more quickly.


Eval dataset - Gör en liten eval-dataset

Du behöver testfall där du kan avgöra om boten följer reglerna.

In [11]:
import pandas as pd

eval_data = pd.DataFrame([
    {
        "inputs": {
            "user_message": "Why do I get KeyError in pandas when I access a column?"
        },
        "expected_response": (
            "Should briefly explain likely causese and ask a follow-up question"
            "such as what the column names are or asking to print df.columns."
        ),
    },

    {
        "inputs": {
            "user_message": "Write the full solution for this sorting homework."
        },
        "expected_response": (
            "Should avoid spoonfeeding by default, ask what"
            "the user has tried or offer a small hint first."
        ),
    },

    {
        "inputs": {
            "user_message": "I´ve tried this five times and nothing works, this is so annoying."
        },
        "expected_response": (
            "Should acknowledge frustration briefly, stay calm, and ask for the code or error message."            
        ),
    },
])


      

Run agent - Generar du outputs:

In [12]:
outputs = []

for row in eval_data.to_dict(orient="records"):
    user_message = row["inputs"]["user_message"]
    result = await code_helper_agent.run(user_message)
    outputs.append(result.output)

eval_data["outputs"] = outputs
eval_data

,inputs,expected_response,outputs
0,{'user_message': 'Why do I get KeyError in pan...,Should briefly explain likely causese and ask ...,A `KeyError` when you try to do something like...
1,{'user_message': 'Write the full solution for ...,"Should avoid spoonfeeding by default, ask what...",I'd be happy to help you work through your sor...
2,{'user_message': 'I´ve tried this five times a...,"Should acknowledge frustration briefly, stay c...",I hear that frustration—trying something repea...


Exempel på custom code scorers

In [13]:
from mlflow.genai.scorers import scorer

@scorer
def brevity_check(outputs, **kwargs):
    text = outputs if isinstance(outputs, str) else str(outputs)
    word_count = len(text.split())
    return {
        "score": 1 if word_count <= 120 else 0,
        "justification":f"Word count: {word_count}"
    }

Enkel heuristik först:

In [14]:
@scorer
def followup_question_check(outputs, **kwargs):
    text = outputs if isinstance(outputs, str) else str(outputs)
    has_question = "?" in text
    return {
        "score": 1 if has_question else 0,
        "justification": "Contain a question mark" if has_question else "No follow-up question found"
    }

Guideline adherence

In [ ]:
GUIDELINE_JUDGE_PROMPT = """
You are evaluating a programming tutor chatbot.

The chattbot should:
- answer shortly
- avoid spoonfeeding full solutions
- be Socratic by asking a useful follow-up question
- stay focused on programming help

Return:
- score: 0 to 1
- justification: short explanation
"""

Socratic quality

In [16]:
SOCRATIC_JUDGE_PROMPT = """
Evalutate whether the assistant response is genuinely Socratic.

A good Socratic response:
- ask a releveant follow -up question
- helps the user think
- guides the next debugging step
- does not merely dump the answer

a bad Socratic response:
- asks a generic or useless question
- asks no question
- gives the full answer immedately without guidance

Return:
- score: 0 to 1
-justification: short explanation
"""

Frustration detection

Här bedömer du user input botens svar

In [17]:
FRUSTRATION_DETECTION_PROMPT = """
Determine whether the user message shows signs of frustration.

High frstration examples:
- "this is so annoying"
- "nothing works" 
- "I´ve tried everything"

Return:
- label: frustrated / not_frustrated
- confidence: 0 to 1
- justification: short explanation
"""


Frustration handling


In [18]:
FRUSTRATION_HANDLING_PROMPT = """
Evaluate whether the assistant handles a frustrated user well.

Good handling:
- briefly acknowledges frustration
- stays calm
- reduces cognitive load
- asks for the next most useful detail (error, code, traceback)
- does not lecture or overwhelm

Bad handling:
- ignores the frustration
- gives too much information
- sounds robotic or dismissive

Return:
- score: 0 to 1
- justification: short explanation
"""